# Lab 3 - Storing chats like ChatGPT (SQLite)

ChatGPT's sidebar is just a **database**. Every conversation is a row; every message is a
row pointing at its conversation. New chat = insert a row. Switch chats = query by id.

This lab builds exactly that with **SQLite** (built into Python - no server, one file on
disk) and wires it to a ChatGPT-style Gradio screen: a list of past chats on the left, the
active conversation on the right, plus new / rename / delete.

```
conversations                messages
+----+-------------+          +----+-----------------+--------+-----------+
| id | title       |          | id | conversation_id | role   | content   |
+----+-------------+          +----+-----------------+--------+-----------+
|  1 | Sensor help |  <-----  |  1 |        1        | user   | "hi..."   |
|  2 | Trip plan   |          |  2 |        1        | assistant | "..."  |
+----+-------------+          +----+-----------------+--------+-----------+
```

> On Colab the `.db` file lives in the session and disappears when the runtime resets. To
> keep it, mount Google Drive and point `DB_PATH` there (one line, shown below).


## Step 0 - Install

In [ ]:
%pip install -q langchain langchain-groq langchain-openai gradio

## Step 1 - Model: Groq first, OpenRouter as a fallback

The chat itself has nothing to do with SQLite - it just needs to be **fast**, or the
"ChatGPT feel" is gone. So the primary is **Groq** (`openai/gpt-oss-120b`, near-instant on
the free tier). If Groq is unreachable or rate-limited we fall through to a couple of free
**OpenRouter** models. One small wrapper gives us `.invoke()` (whole reply) and `.stream()`
(token by token - used by the Gradio screen in Step 7).

In [ ]:
import os

def load_key(name, required=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: from Colab secret"); return v
    except Exception:
        pass
    if os.getenv(name):
        print(f"{name}: from environment"); return os.environ[name]
    from getpass import getpass
    tail = "" if required else "  (optional - press Enter to skip)"
    return getpass(f"Paste {name}{tail}: ").strip()

GROQ_API_KEY       = load_key("GROQ_API_KEY")
OPENROUTER_API_KEY = load_key("OPENROUTER_API_KEY", required=False)

from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

MODELS = [ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY,
                   temperature=0.4, max_retries=2)]
if OPENROUTER_API_KEY:
    for _m in ["nvidia/nemotron-3-super-120b-a12b:free", "minimax/minimax-m2.7:free"]:
        MODELS.append(ChatOpenAI(model=_m, base_url="https://openrouter.ai/api/v1",
                                 api_key=OPENROUTER_API_KEY, temperature=0.4, max_retries=1))

def _name(m):
    return getattr(m, "model_name", getattr(m, "model", "?"))

class LLM:
    """Try each backend in order. .invoke() -> full text; .stream() -> text chunks."""
    def invoke(self, messages):
        err = None
        for m in MODELS:
            try:
                return m.invoke(messages).content
            except Exception as e:
                err = e; print(f"  ({_name(m)} failed: {str(e)[:70]})")
        raise err

    def stream(self, messages):
        err = None
        for m in MODELS:
            try:
                any_chunk = False
                for chunk in m.stream(messages):
                    any_chunk = True
                    if chunk.content:
                        yield chunk.content
                if any_chunk:
                    return
            except Exception as e:
                err = e; print(f"  ({_name(m)} stream failed: {str(e)[:70]})")
        raise err

llm = LLM()
print("models:", [_name(m) for m in MODELS])
print(llm.invoke("Reply with one word: ready"))

## Step 2 - Create the database

Two tables. `messages.conversation_id` links each message to its conversation. `PRAGMA
foreign_keys` + `ON DELETE CASCADE` means deleting a conversation also deletes its
messages.

In [ ]:
import sqlite3, datetime

DB_PATH = "chat_history.db"
# To persist on Colab:  from google.colab import drive; drive.mount('/content/drive')
#                        DB_PATH = "/content/drive/MyDrive/chat_history.db"

def connect():
    con = sqlite3.connect(DB_PATH)
    con.execute("PRAGMA foreign_keys = ON")
    return con

with connect() as con:
    con.executescript("""
        CREATE TABLE IF NOT EXISTS conversations (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            title      TEXT NOT NULL DEFAULT 'New chat',
            created_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS messages (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id INTEGER NOT NULL REFERENCES conversations(id) ON DELETE CASCADE,
            role            TEXT NOT NULL,
            content         TEXT NOT NULL,
            created_at      TEXT NOT NULL
        );
    """)
print("Database ready at", DB_PATH)

## Step 3 - The CRUD functions (create / read / update / delete)

In [ ]:
def now():
    return datetime.datetime.now().isoformat(timespec="seconds")

def new_conversation(title="New chat"):
    with connect() as con:
        cur = con.execute("INSERT INTO conversations(title, created_at) VALUES (?, ?)", (title, now()))
        return cur.lastrowid

def list_conversations():
    with connect() as con:
        return con.execute("SELECT id, title FROM conversations ORDER BY id DESC").fetchall()

def add_message(conversation_id, role, content):
    with connect() as con:
        con.execute("INSERT INTO messages(conversation_id, role, content, created_at) VALUES (?, ?, ?, ?)",
                    (conversation_id, role, content, now()))

def get_messages(conversation_id):
    with connect() as con:
        return con.execute("SELECT role, content FROM messages WHERE conversation_id = ? ORDER BY id",
                           (conversation_id,)).fetchall()

def rename_conversation(conversation_id, title):
    with connect() as con:
        con.execute("UPDATE conversations SET title = ? WHERE id = ?", (title, conversation_id))

def delete_conversation(conversation_id):
    with connect() as con:
        con.execute("DELETE FROM conversations WHERE id = ?", (conversation_id,))

# quick smoke test
cid = new_conversation("Test chat")
add_message(cid, "user", "hello")
add_message(cid, "assistant", "hi there")
print("conversations:", list_conversations())
print("messages in", cid, ":", get_messages(cid))
delete_conversation(cid)
print("after delete:", list_conversations())

## Step 4 - Auto-title a conversation from its first message (like ChatGPT does)

In [ ]:
def auto_title(first_user_message):
    prompt = ("Give a 3-5 word title for a chat that starts with the message below. "
              "Reply with ONLY the title - no quotes, no explanation, one line.\n\n"
              + first_user_message)
    raw = llm.invoke(prompt).strip()
    title = raw.splitlines()[0].strip().strip('"').lstrip("#").strip()   # first line only
    return title[:50] or "New chat"

print(auto_title("Which air quality sensor should I use with an ESP32?"))

## Step 5 - `send()` - one chat turn, fully persisted

Load the conversation's history from the DB, send it to the model, then **save both** the
user message and the reply. Nothing lives only in memory.

In [ ]:
def send(conversation_id, user_text):
    history = get_messages(conversation_id)                      # from DB
    lc_messages = [{"role": r, "content": c} for r, c in history]
    lc_messages.append({"role": "user", "content": user_text})

    reply = llm.invoke(lc_messages)

    add_message(conversation_id, "user", user_text)             # persist
    add_message(conversation_id, "assistant", reply)

    if len(history) == 0:                                        # first turn -> title it
        rename_conversation(conversation_id, auto_title(user_text))
    return reply

cid = new_conversation()
print(send(cid, "In one sentence, what is an air quality sensor?"))
print(send(cid, "Name one cheap model."))
print("\nStored title:", [t for i, t in list_conversations() if i == cid][0])
print("Stored turns :", len(get_messages(cid)))

## Step 6 - The point: it survives a "restart"

Every function opens a **fresh connection** to the file. Close everything, reconnect, and
the conversations are still there - because they were never in a Python variable, they were
on disk.

In [ ]:
# simulate a fresh process: a brand-new connection, zero in-memory state carried over
fresh = sqlite3.connect(DB_PATH)
rows = fresh.execute("SELECT id, title FROM conversations ORDER BY id DESC").fetchall()
fresh.close()
print("After a simulated restart, the database still has:")
for i, t in rows:
    print(f"  #{i}  {t}  ({len(get_messages(i))} messages)")

## Step 7 - ChatGPT-style Gradio screen

Left: the list of saved chats + New / Delete. Right: the active conversation. Selecting a
chat loads it from the DB.

Sending a message is a **generator**: it shows your message right away, then streams the
reply token by token (`llm.stream(...)`), and only when the reply is complete does it write
**both** messages to SQLite and auto-title a brand-new chat. That streaming is what makes
it feel like ChatGPT instead of a 20-second freeze.

In [ ]:
import gradio as gr

def refresh_list():
    convs = list_conversations()
    choices = [(t, i) for i, t in convs]           # (label, value)
    return gr.update(choices=choices, value=(choices[0][1] if choices else None))

def load_conversation(conversation_id):
    if not conversation_id:
        return []
    return [{"role": r, "content": c} for r, c in get_messages(conversation_id)]

def do_send(conversation_id, user_text, chat_display):
    if not user_text.strip():
        yield chat_display, "", gr.update(), conversation_id
        return
    if not conversation_id:
        conversation_id = new_conversation()

    history = get_messages(conversation_id)                       # from DB
    lc_messages = [{"role": r, "content": c} for r, c in history]
    lc_messages.append({"role": "user", "content": user_text})

    # 1. echo the user's message + an empty assistant bubble immediately
    view = [{"role": r, "content": c} for r, c in history]
    view.append({"role": "user", "content": user_text})
    view.append({"role": "assistant", "content": ""})
    yield view, "", gr.update(), conversation_id

    # 2. stream the reply into that bubble
    reply = ""
    for piece in llm.stream(lc_messages):
        reply += piece
        view[-1]["content"] = reply
        yield view, "", gr.update(), conversation_id

    # 3. now persist both messages, and title the chat if it is new
    add_message(conversation_id, "user", user_text)
    add_message(conversation_id, "assistant", reply)
    if len(history) == 0:
        rename_conversation(conversation_id, auto_title(user_text))
    yield load_conversation(conversation_id), "", refresh_list(), conversation_id

def do_new():
    cid = new_conversation()
    return cid, [], refresh_list()

def do_delete(conversation_id):
    if conversation_id:
        delete_conversation(conversation_id)
    convs = list_conversations()
    new_cid = convs[0][0] if convs else None
    return new_cid, load_conversation(new_cid), refresh_list()

with gr.Blocks(title="Lab 3 - Chat Store") as demo:
    gr.Markdown("# Lab 3 - Chats stored in SQLite")
    current = gr.State(None)
    with gr.Row():
        with gr.Column(scale=1):
            new_btn = gr.Button("New chat", variant="primary")
            chat_list = gr.Radio(label="Your chats", choices=[], interactive=True)
            del_btn = gr.Button("Delete selected chat", variant="stop")
        with gr.Column(scale=3):
            chatbox = gr.Chatbot(height=430)
            msg = gr.Textbox(label="Message", placeholder="Type and press Enter")

    demo.load(refresh_list, outputs=[chat_list])
    chat_list.change(lambda cid: (cid, load_conversation(cid)), inputs=chat_list, outputs=[current, chatbox])
    new_btn.click(do_new, outputs=[current, chatbox, chat_list])
    del_btn.click(do_delete, inputs=current, outputs=[current, chatbox, chat_list])
    msg.submit(do_send, [current, msg, chatbox], [chatbox, msg, chat_list, current])

demo.launch(debug=False)

## Recap

- A chat app is a **database with a chat UI on top**. Two tables did it.
- `conversations` = the sidebar list; `messages` = the transcript, linked by `conversation_id`.
- Each turn reads history from disk, **streams** the model's reply, then writes both
  messages back - so refreshing, switching chats, or restarting loses nothing.
- Speed matters for the "ChatGPT feel": a **fast primary model** (Groq) plus **token
  streaming** is the difference between instant and a multi-second freeze.
- Lab 4 adds a third table: things the assistant should **remember about you** across all
  of these chats.

### Exercises
1. Add a `updated_at` column and sort the sidebar by most-recently-used instead of newest.
2. Add a search box that runs `SELECT ... WHERE content LIKE ?` over messages.
3. Point `DB_PATH` at Google Drive and confirm the chats survive a full runtime restart.
4. Only the last N messages really need to go to the model - cap the history in the turn
   handler at 12 and confirm long chats stay fast.
